<a href="https://colab.research.google.com/github/asmaatefomran/generative-ai-tasks/blob/main/Task4_simple_rag_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Task 4: Simple RAG System**

Install packages

In [1]:
!pip install -q langchain-community langchain-huggingface langchain-text-splitters chromadb pypdf sentence-transformers groq


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/20

Imports


In [2]:
import os
from google.colab import userdata

from groq import Groq

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma


/tmp/ipykernel_4392/2505230032.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Load your Groq API key

In [3]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

client = Groq(api_key=os.environ["GROQ_API_KEY"])


Upload your PDF

In [4]:
from google.colab import files

uploaded = files.upload()


Saving Coding-Challenge-Full-Stack-Developer (1).pdf to Coding-Challenge-Full-Stack-Developer (1).pdf


In [5]:
pdf_file = list(uploaded.keys())[0]

print("Loaded:", pdf_file)


Loaded: Coding-Challenge-Full-Stack-Developer (1).pdf


Load the PDF

In [6]:
loader = PyPDFLoader(pdf_file)

documents = loader.load()

print("Pages:", len(documents))


Pages: 4


Split the document

In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))


Chunks: 16


Create embeddings

In [9]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Create the vector database

In [10]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector database created.")


Vector database created.


Create the retriever

In [11]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)


Ask a question

In [12]:
question = input("Ask a question about the PDF: ")

retrieved_docs = retriever.invoke(question)

print("Retrieved documents:", len(retrieved_docs))


Ask a question about the PDF: What is this pdf about?
Retrieved documents: 3


Send retrieved information to Groq

In [13]:
context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

prompt = f"""
Answer the question using ONLY the provided context.

If the answer is not available in the context, say:
"I could not find the answer in the document."

Context:
{context}

Question:
{question}

Answer:
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

print("\nAnswer:")
print(response.choices[0].message.content)



Answer:
The PDF is a coding‑challenge brief for Octo Education. It describes a task to design and implement a “Program Designer” API for their education‑management software—covering program validation, API contracts, a simulation endpoint, and the required submission checklist.


                 PDF
                  ↓
             Load PDF
                  ↓
            Split into chunks
                  ↓
              Embeddings
                  ↓
            Vector Database
                  ↓
              Retriever
                  ↓
           Relevant chunks
                  ↓
                Groq
                  ↓
              Answer
